In [0]:
dbutils.widgets.removeAll()

In [0]:
%run ./3.1.Transform_SFC_Indicadores

In [0]:
from pyspark.sql import functions as F

tabla_destino = "catalog_au.silver.sfc_indicadores_v2"

print("Tabla destino:", tabla_destino)
print("Registros transformados:", df_silver_final.count())

display(
    df_silver_final
    .groupBy("estado_validacion")
    .count()
    .orderBy("estado_validacion")
)

In [0]:
%sql

SELECT COUNT(*) AS registros_actuales
FROM catalog_au.silver.sfc_indicadores_v2;

In [0]:
%sql

SELECT
    COUNT(*) AS registros_actuales
FROM catalog_au.silver.sfc_indicadores_v2;

In [0]:
df_silver_final.createOrReplaceTempView(
    "vw_sfc_indicadores_transformados"
)

print("Vista temporal creada correctamente.")

In [0]:
%sql

MERGE INTO catalog_au.silver.sfc_indicadores_v2 AS destino

USING vw_sfc_indicadores_transformados AS origen

ON destino.id_indicador = origen.id_indicador

WHEN MATCHED THEN UPDATE SET
    destino.codigo_entidad = origen.codigo_entidad,
    destino.nombre_entidad = origen.nombre_entidad,
    destino.codigo_concepto = origen.codigo_concepto,
    destino.nombre_concepto = origen.nombre_concepto,
    destino.fecha_corte = origen.fecha_corte,
    destino.anio = origen.anio,
    destino.mes = origen.mes,
    destino.valor_indicador = origen.valor_indicador,
    destino.unidad_medida = origen.unidad_medida,
    destino.archivo_origen = origen.archivo_origen,
    destino.fecha_ingesta = origen.fecha_ingesta,
    destino.fecha_transformacion = origen.fecha_transformacion,
    destino.estado_validacion = origen.estado_validacion

WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql

SELECT
    anio,
    estado_validacion,
    COUNT(*) AS total_registros
FROM catalog_au.silver.sfc_indicadores_v2
GROUP BY anio, estado_validacion
ORDER BY anio, estado_validacion;

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_indicador) AS ids_unicos,
    COUNT(DISTINCT codigo_entidad) AS entidades,
    COUNT(DISTINCT codigo_concepto) AS conceptos,
    SUM(
        CASE
            WHEN estado_validacion = 'VALIDO' THEN 1
            ELSE 0
        END
    ) AS indicadores_validos,
    SUM(
        CASE
            WHEN estado_validacion = 'VALIDO'
                 AND valor_indicador IS NULL
            THEN 1 ELSE 0
        END
    ) AS validos_sin_valor
FROM catalog_au.silver.sfc_indicadores_v2;